# Algoritmo genético

DEsarrollo incio

In [1]:
import pandas as pd
import random
import numpy as np

import time
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split

from deap import base, creator, tools, algorithms

## Datos iniciales

In [2]:
Entrenamiento = pd.read_csv("Entrenamiento.csv")
Ajuste = pd.read_csv("Ajuste.csv")

In [3]:
Entrenamiento = Entrenamiento.replace(np.nan, 0)
Ajuste = Ajuste.replace(np.nan, 0)

In [4]:
Entrenamiento['Label'] = Entrenamiento["Label"].replace({"s":1, "b":0}).astype(int)
Ajuste["Label"] = Ajuste["Label"].replace({"s":1, "b":0}).astype(int)

Para que el estudio de los datos no sea tan pesado, se va a coger un subgrupo más pequeño para desarrollo el algortimo genético.

In [5]:
Entrenamiento.sample(n=50000)

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label,KaggleSet,KaggleWeight,DER_mass_MMC_missing
185747,285747,0.331807,-0.889527,-0.695011,1.778747,0.587336,1.144836,-0.506736,-2.019518,1.431294,...,1.295588,0.260876,0.580218,0.481900,1.347775,0.000461,1,t,0.001503,1
55006,155006,-0.162570,0.948671,0.209588,0.774422,0.000000,0.000000,0.000000,-0.732013,1.360153,...,-1.177143,0.000000,0.000000,0.000000,0.386428,0.326117,0,t,1.069553,1
246252,346252,0.596697,-0.106707,0.907312,0.662174,1.538998,1.110543,-1.461166,-0.126579,0.722034,...,1.383270,-0.933721,1.570015,0.207813,0.887735,0.000461,1,t,0.001503,1
225102,325102,0.000000,0.889761,-2.595001,-0.117411,0.000000,0.000000,0.000000,-2.054005,0.720836,...,0.000000,0.000000,0.000000,0.000000,-1.189337,0.559012,0,t,1.833369,0
226463,326463,-1.628400,0.373055,-1.135515,0.279336,0.000000,0.000000,0.000000,-1.094762,0.497747,...,0.519137,0.000000,0.000000,0.000000,0.587224,0.756905,0,t,2.482390,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78879,178879,0.000000,1.174999,0.363980,-1.389815,0.000000,0.000000,0.000000,-0.253030,-0.753859,...,0.000000,0.000000,0.000000,0.000000,-1.189337,0.656658,0,t,2.153614,0
196205,296205,0.000000,0.528387,-0.704180,-1.019821,0.000000,0.000000,0.000000,-0.292626,-0.325042,...,0.000000,0.000000,0.000000,0.000000,-1.189337,1.515986,0,t,4.971917,0
181399,281399,-0.712991,-0.246021,-0.627792,-0.214020,0.000000,0.000000,0.000000,0.485241,1.018611,...,-0.430471,0.000000,0.000000,0.000000,0.542529,0.443598,0,t,1.454848,1
30960,130960,0.680099,0.975350,0.959958,0.245195,0.000000,0.000000,0.000000,0.088005,1.141089,...,0.000000,0.000000,0.000000,0.000000,-1.189337,0.312428,0,t,1.024658,1


Al igual que en el desarrollo del MLP se va a estratificar el conjunto de entrenamiento para tener un grupo de validación equilibrando los pesos para que no afecte más el fondo que la señal a la hora de entrenar.

In [6]:
Entrenamiento, Validacion = train_test_split(Entrenamiento, test_size=0.2,stratify=Entrenamiento["Label"], random_state=42)

In [7]:
def equilibrar(df):
    w = df["KaggleWeight"] / df.groupby("Label")["KaggleWeight"].transform("sum") * (len(df) / 2)
    return w.to_numpy()

In [8]:
# Entrenamiento DER / DER + PRI

Y_E = Entrenamiento["Label"]
W_E = equilibrar(Entrenamiento)
Entrenamiento = Entrenamiento.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight", "DER_mass_MMC_missing"])

In [9]:
Y_V = Validacion["Label"]
W_V = equilibrar(Validacion)
Validacion = Validacion.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight", "DER_mass_MMC_missing"])

In [10]:
Y_A = Ajuste["Label"]
W_A = equilibrar(Ajuste)
Ajuste = Ajuste.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight", "DER_mass_MMC_missing"])

## MLP

In [11]:
def creacion_red(shape_input, num_neuronas, dropout = 0):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    capas = [tf.keras.layers.Input(shape=(shape_input,))]

    for n in num_neuronas:
        capas.append(tf.keras.layers.Dense(n, activation="relu", kernel_initializer="he_normal"))
        if dropout > 0:
            capas.append(tf.keras.layers.Dropout(dropout))

    capas.append(tf.keras.layers.Dense(1, activation="sigmoid"))  

    return tf.keras.Sequential(capas)

In [12]:
# 1. Compilar para modelo_PRI y modelo_DER (modelo==red)
def aprendizaje(modelo):
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        weighted_metrics=[tf.keras.metrics.AUC(name="auc")]
    )

# 2. Criterio de parada para ambos el mismo, ya que al final se desean comparar los resultados.
parada = tf.keras.callbacks.EarlyStopping(
    monitor = "val_loss",
    patience = 2,                  # máx épocas seguidas sin mejorar
    restore_best_weights=True
)

In [13]:
# Entrenamiento de MLP
def entrenar(Entrenamiento, modelo, Y_E, W_E, Validacion, Y_V, W_V):

    aprendizaje(modelo)

    historial = modelo.fit(
        Entrenamiento, Y_E,
        sample_weight = W_E,
        validation_data = (Validacion, Y_V, W_V),
        epochs = 15,                                   # máx iteraciones de entrenamiento
        batch_size = 1024,                               # Número de eventos que se procesan antes de cada actualización de pesos
        callbacks = [parada],
        verbose = 0
    )

    return historial

In [14]:
def calcular_ams(y, prob, pesos, umbral):           # el AMS permite ver cómo de acertado fue (cuanto más alto mejor)
    c = 10
    seleccionados = prob >= umbral                  # Genera un vector true false, siendo true si el evento es cogido por señal, es decir pasa el filtro del umbral
    s = pesos[seleccionados & (y == 1)].sum()       # Suma la cantidad de eventos que sean señal tanto en el modelo como de verdad
    b = pesos[seleccionados & (y == 0)].sum()

    AMS = np.sqrt(2*((s+b+c)*np.log(1+(s/(b+c)))-s))
    return AMS

def buscar_mejor_umbral(y, prob, pesos):
    umbrales = np.quantile(prob, np.linspace(0.50, 0.99, 200))
    lista_ams = [calcular_ams(y, prob, pesos, u) for u in umbrales]
    mejor = np.argmax(lista_ams)
    return umbrales[mejor], lista_ams[mejor]

## Desarrollo del algoritmo genético 

Se va a desarrollar con la librería DEAP (Distributed Evolutionary Algorithms in Python)

In [15]:
keras.utils.set_random_seed(42)

variables = Entrenamiento.columns.tolist()
cromosoma = [random.randint(0, 1) for _ in range(len(variables))]

Al final lo que representa cada uno de los cromosomas son las variables que se contabilizan para ver si el AMs lograda es mejor que con él.

In [16]:
def columnas(cromosoma):
    # Se queda con las variables cuyo valor es 1
    i = 0 
    cols = []

    for c in cromosoma:
        if c == 1:
            cols.append(variables[i])
        i +=1 

    return cols

Función de aptitud (Fitness). Para hacer más simple la manipulación de datos se va a ir guardando en un csv

In [17]:
cache = {}
LAMBDA = 0.01   # penalización por cada variable usada

# Crea el CSV con la cabecera (las evaluaciones se irán añadiendo debajo)
pd.DataFrame(columns=["cromosoma", "n_variables", "ams", "aptitud"]).to_csv("evaluaciones_cromosomas.csv", index=False)

def evaluar(cromosoma):
    # 1. Si ya lo evaluamos, devolvemos el resultado guardado
    clave = tuple(cromosoma)
    if clave in cache:
        return cache[clave]

    # 2. Sin variables no hay modelo
    cols = columnas(cromosoma)
    if len(cols) == 0:
        return (0.0,)

    # 3. MLP pequeño
    modelo = creacion_red(len(cols), [32, 16])    
    inicio = time.perf_counter()
    historial = entrenar(Entrenamiento[cols], modelo, Y_E, W_E, Validacion[cols], Y_V, W_V)    
    fin = time.perf_counter()

    # 4. AMS en b y aptitud con penalización
    probabilidades = modelo.predict(Ajuste[cols], verbose=0).ravel()
    umbral, ams = buscar_mejor_umbral(Y_A, probabilidades, W_A)
    n_cols = sum(cromosoma)         # numero de columnas/variables que usa el cromosoma
    aptitud = ams - LAMBDA * n_cols

    # 5. Guardar en la caché y en el CSV
    cache[clave] = (aptitud,)
    fila = {"crosoma": "".join(map(str, cromosoma)), "n_variables": n_cols, "ams": ams, "aptitud": aptitud}
    pd.DataFrame([fila]).to_csv("evaluaciones_cromosomas.csv", mode="a", header=False, index=False)

    return (aptitud,)   # DEAP exige una tupla

CONFIGURACION DEL DEAP

In [18]:
creator.create("FitnessMax", base.Fitness, weights=(1.0,))          # maximizar
creator.create("Individual", list, fitness=creator.FitnessMax)     # individuo = lista

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=len(variables))
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluar)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=1/len(variables))
toolbox.register("select", tools.selTournament, tournsize=3)

### Ejecutar el AG

In [19]:
poblacion = toolbox.population(n=30)
poblacion[0] = creator.Individual([1] * len(variables))  # individuo de referencia

hof = tools.HallOfFame(5)
stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register("max", np.max)
stats.register("mean", np.mean)
stats.register("std", np.std)

poblacion, historial = algorithms.eaSimple(
    poblacion, toolbox,
    cxpb=0.7, mutpb=0.2,
    ngen=20,                     # ⚠️ prueba: 3. Real: 15
    stats=stats, halloffame=hof, verbose=True)

pd.DataFrame(historial).to_csv("historial.csv", index=False)


gen	nevals	max    	mean   	std    
0  	30    	338.733	311.267	20.5215
1  	26    	339.716	326.411	9.33882
2  	26    	339.274	333.539	4.68923
3  	26    	340.757	337.226	2.43746
4  	26    	340.757	338.319	1.00763
5  	26    	340.757	338.757	1.10347
6  	26    	340.757	338.94 	0.931249
7  	26    	340.757	339.158	0.988057
8  	26    	340.757	339.35 	1.13438 
9  	26    	340.757	339.854	1.02821 
10 	26    	340.757	340.087	1.08617 
11 	26    	340.757	340.24 	1.12075 
12 	25    	340.757	340.327	1.2632  
13 	26    	340.757	340.279	1.12781 
14 	25    	340.757	340.327	1.2632  
15 	26    	340.757	340.598	0.558727
16 	26    	340.757	340.279	1.12781 
17 	25    	340.757	340.327	1.2632  
18 	26    	340.757	340.598	0.558727
19 	25    	340.757	340.757	5.68434e-14
20 	20    	340.757	340.475	0.7919     


### Análisis de los mejores resultados

In [20]:
mejores = [{"n_variables": sum(ind), "aptitud": ind.fitness.values[0], "variables": ", ".join(columnas(ind))}
           for ind in hof]

df_mejores = pd.DataFrame(mejores)

# Añadir la columna ams en la posición 1 (la segunda)
df_mejores.insert(1, "ams", df_mejores["aptitud"] + LAMBDA * df_mejores["n_variables"])

df_mejores.to_csv("mejores_cromosomas.csv", index=False)
print(df_mejores[["n_variables","ams","aptitud"]])

   n_variables         ams     aptitud
0           25  341.006934  340.756934
1           24  340.655115  340.415115
2           24  340.504890  340.264890
3           26  340.462761  340.202761
4           24  340.415510  340.175510
